# Customer Support Copilot: LLM Components
## Instructor-led ticket triage and policy-agent notebook

This notebook covers privacy-safe ticket triage, policy retrieval (RAG), and a bounded refund-agent tool loop using the local mock LLM. No hosted model, API key, or network call is needed.

### Before class
1. Install packages with `pip install -r requirements.txt`.
2. Run `notebooks/Data_Cleaning_and_ML.ipynb` first to create the cleaned data and `data/instamart.db`.
3. In VS Code, select the **Python (Instamart Copilot)** kernel and run this notebook from the repository or its `notebooks/` folder.

In [1]:
from pathlib import Path
import sqlite3
import sys
import pandas as pd
ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "generate_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the project folder so generate_data.py can be found.")
sys.path.insert(0, str(ROOT))
db_path = ROOT / "data" / "instamart.db"
if not db_path.exists():
    raise FileNotFoundError("Run notebooks/Data_Cleaning_and_ML.ipynb first to create data/instamart.db.")
print("Project root:", ROOT)
print("SQLite database:", db_path)

Project root: d:\Barathm\Desktop\Barath\IIT Patna\instamart_customer_copilot
SQLite database: d:\Barathm\Desktop\Barath\IIT Patna\instamart_customer_copilot\data\instamart.db


## 6. Part C — support ticket triage

This pipeline demonstrates:
1. input guardrails for empty and overlong tickets;
2. ordered PII redaction (email → phone → order ID);
3. retry logic for simulated transient errors;
4. strict category validation and a keyword fallback;
5. token budgeting and human-review routing.

The exact user message sent to the mock classifier is the redacted ticket text only.

In [2]:
from src.triage import run_pipeline, FEW, ZERO
triage_results, triage_summary = run_pipeline(prompt=FEW, token_budget=100000)
print(triage_summary)
display(triage_results.head(10))
display(triage_results.status.value_counts().rename_axis("status").reset_index(name="tickets"))

retry ticket=T0002 retry=0 wait=0.052993s
retry ticket=T0116 retry=0 wait=0.045313s
retry ticket=T0153 retry=0 wait=0.048438s
retry ticket=T0039 retry=0 wait=0.047790s
retry ticket=T0152 retry=0 wait=0.054206s
retry ticket=T0043 retry=0 wait=0.053459s
retry ticket=T0111 retry=0 wait=0.056152s
retry ticket=T0155 retry=0 wait=0.046087s
retry ticket=T0100 retry=0 wait=0.050274s
retry ticket=T0068 retry=0 wait=0.045372s
retry ticket=T0135 retry=0 wait=0.047733s
retry ticket=T0009 retry=0 wait=0.051317s
retry ticket=T0067 retry=0 wait=0.045332s
retry ticket=T0106 retry=0 wait=0.047485s
retry ticket=T0097 retry=0 wait=0.053124s
retry ticket=T0003 retry=0 wait=0.051812s
retry ticket=T0019 retry=0 wait=0.047756s
retry ticket=T0086 retry=0 wait=0.052366s
retry ticket=T0065 retry=0 wait=0.055118s
{'tokens': 13768, 'classified_tickets': 147, 'accuracy': 1.0, 'tokens_per_classified_ticket': 93.65986394557824, 'confusion_matrix': predicted_category  billing  delivery  fraud  technical
true_category

,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,Dormant,HIGH,security_team,92
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,88
2,T0095,C1087,classified_json,technical,Occasional,NORMAL,support_queue,90
3,T0159,C1172,skipped_empty,,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,Occasional,HIGH,security_team,92
5,T0124,C0139,classified_json,fraud,Frequent,HIGH,security_team,92
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,84
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,87
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,88
9,T0002,C0881,classified_json,delivery,Occasional,NORMAL,support_queue,86


,status,tickets
0,classified_json,135
1,classified_keyword,12
2,flagged_invalid_output,9
3,skipped_empty,4
4,flagged_long,4


### Discussion: privacy and safety
- Redaction is a defence-in-depth step, not a guarantee that every sensitive detail is removed.
- Why are invalid categories sent to human review instead of guessed?
- Why should fraud override ordinary customer-segment routing?
- Why do empty/long tickets need to be blocked before any model call?

## 7. Part D — policy RAG

The retrieval layer chunks policy documents, fits a TF-IDF vectorizer and ranks chunks by cosine similarity. It passes retrieved chunks to the offline policy assistant, which should answer only from the context and fall back when a question is not covered.

In [3]:
from src.policy_rag import PolicyRAG, evaluate
rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
print(rag.answer("What delivery time makes a completed order eligible for a full refund?"))
policy_eval = evaluate(rag)
display(policy_eval)
print("Expected-document hits:", int(policy_eval.retrieved_doc.sum()))
print("Expected-fact hits:", int(policy_eval.fact_found.sum()))
print("Fallbacks:", int(policy_eval.fallback.sum()))

Policy chunks: 15


,source,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ..."
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours...."


{'answer': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.', 'sources': ['refund_policy.txt', 'privacy_policy.txt'], 'retrieved': [{'source': 'refund_policy.txt', 'score': 0.35464416504570484, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.31425507733397284, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'privacy_policy.txt', 'score': 0.15759666156595936, 'text': 'Delivery addresses are deleted from active systems 180 days after the last order. A customer can ask for account deletion and the request is c

,question,expected_doc,retrieved_doc,fact_found,fallback
0,What delivery time makes a completed order eli...,refund_policy.txt,True,True,False
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,False
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,False
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,False
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,False
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,False
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,True,False
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,False
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,False


Expected-document hits: 12
Expected-fact hits: 12
Fallbacks: 2


## 8. Part D — refund tools and safe stopping

The refund agent demonstrates a tool loop: look up an order in SQLite, retrieve policy, calculate the eligible refund, then issue it or escalate to a human. In this teaching kit the ledger is a local JSONL file; no money moves.

Before running agent demonstrations, inspect `src/refund_agent.py` and ask students to trace how a tool name maps to a Python function.

In [4]:
from src.refund_agent import get_order, search_policy, calculate_refund, TOOL_MAP, run_agent
print("Tools:", sorted(TOOL_MAP))
print("Example order lookup:")
with sqlite3.connect(db_path) as con:
    example_order = pd.read_sql_query(
        "SELECT order_id FROM orders WHERE status='completed' AND delivery_minutes > 45 LIMIT 1", con
    ).iloc[0, 0]
print(example_order, get_order(example_order))
print("Refund calculation:", calculate_refund(250.0, 52))
print("Timeout safety demo:", run_agent(f"Refund order {example_order}", scenario_id="timeout-demo", time_limit_seconds=0))

Tools: ['calculate_refund', 'escalate_to_human', 'get_order', 'issue_refund', 'search_policy']
Example order lookup:
ORD-100048 {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65}
Refund calculation: {'refund_inr': 250.0}
Timeout safety demo: {'status': 'FALLBACK_TIMEOUT', 'final_answer': 'FALLBACK_TIMEOUT: hand off to a human agent. Time limit reached.', 'steps': [], 'llm_calls': 0, 'tokens_used': 0}


## 9. Review generated artifacts

Review the CSVs and model file produced across both notebooks. To rebuild from source, rerun the documented commands in the README and then run both notebooks in order.

**Important:** This teaching kit is an instructional implementation. If the course grades exact numeric checkpoints from the original brief, run every checkpoint and reconcile differences; do not hardcode expected answers.

In [5]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())

\ndata\clean
['customers.csv', 'orders.csv', 'products.csv']
\noutputs
['agent_runs.jsonl', 'customer_segments.csv', 'lapse_model.joblib', 'refund_ledger.jsonl', 'retention_list.csv', 'triage_budget_demo.csv', 'triage_results.csv']
\nstate
['.gitkeep', 'S1-crash-resume.json', 'S1-token-limit.json', 'scenario-S1.json', 'scenario-S2.json', 'scenario-S3.json', 'scenario-S4.json', 'scenario-S5.json', 'scenario-S6.json', 'scenario-S7.json']
Database exists: True
